# DT Supply — Category-Wise Product Scraper

This notebook scrapes products **category by category, one product at a time**.

For every product it:
1. Gets the product details from the DT Supply API.
2. Downloads the product image into the local `data/images/` folder.
3. Immediately writes the complete product record into `data/dtsupply_products.csv`.
4. Flushes the CSV after every product, so progress is not lost if the notebook stops.
5. Resumes automatically using `product_id`.

### Output structure

```text
data/
├── dtsupply_products.csv
└── images/
    ├── 12345.webp
    ├── 12346.webp
    └── ...
```

The CSV contains the product information plus the **local image path** and original image URL.


In [1]:
!pip install -q requests


In [2]:
import csv
import os
import time
from urllib.parse import urlparse

import requests

# ============================================================
# CONFIGURATION
# ============================================================

BASE_API = "https://api.dtsupply.us/api/Products"

CATEGORIES_URL = f"{BASE_API}/get-all-categories"
PRODUCTS_BY_CATEGORY_URL = f"{BASE_API}/get-products-by-category"

# Everything is stored inside the data folder.
DATA_DIR = "data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
OUTPUT_CSV = os.path.join(DATA_DIR, "dtsupply_products.csv")

REQUEST_DELAY = 0.3
IMAGE_DELAY = 0.2
TIMEOUT = 30
RETRIES = 3

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

FIELDNAMES = [
    "category",
    "category_id",
    "product_id",
    "sku",
    "title",
    "description",
    "price",
    "in_stock",
    "product_url",
    "image_url",
    "image_local",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)

print("Setup complete.")
print(f"CSV: {os.path.abspath(OUTPUT_CSV)}")
print(f"Images: {os.path.abspath(IMAGE_DIR)}")


Setup complete.
CSV: D:\Repositries\100-Days-of-Machine-Learning-CampusX\Day-18-WebScraping\Scrap 2\data\dtsupply_products.csv
Images: D:\Repositries\100-Days-of-Machine-Learning-CampusX\Day-18-WebScraping\Scrap 2\data\images


In [3]:
def get_json(url, params=None):
    """Fetch JSON from the API with retries."""
    for attempt in range(1, RETRIES + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=TIMEOUT,
            )
            response.raise_for_status()
            time.sleep(REQUEST_DELAY)
            return response.json()

        except (requests.RequestException, ValueError) as e:
            print(
                f"    [API warning] attempt "
                f"{attempt}/{RETRIES}: {e}"
            )

            if attempt < RETRIES:
                time.sleep(attempt)

    return None


def format_price(price):
    if price is None or price == 0:
        return "Get a Quote for price"

    try:
        return f"${float(price):,.2f}"
    except (ValueError, TypeError):
        return str(price)


def specs_to_text(specs):
    """Convert the API specs dictionary into readable text."""
    if not specs:
        return ""

    if isinstance(specs, dict):
        return "; ".join(
            f"{key}: {value}"
            for key, value in specs.items()
        )

    return str(specs)


def safe_filename(value):
    value = str(value or "").strip()

    return "".join(
        char if char.isalnum() or char in "-_."
        else "_"
        for char in value
    )


def get_image_extension(response, image_url):
    content_type = (
        response.headers.get("Content-Type") or ""
    ).lower()

    if "webp" in content_type:
        return ".webp"
    if "png" in content_type:
        return ".png"
    if "jpeg" in content_type or "jpg" in content_type:
        return ".jpg"
    if "gif" in content_type:
        return ".gif"
    if "avif" in content_type:
        return ".avif"

    path = urlparse(image_url).path.lower()

    for extension in (
        ".webp",
        ".png",
        ".jpg",
        ".jpeg",
        ".gif",
        ".avif",
    ):
        if path.endswith(extension):
            return extension

    return ".webp"


In [4]:
def download_image(image_url, product_id, product_url=""):
    """Download a product image locally.

    The product page is used as the Referer when available.
    This helps with CDNs that block direct/hotlink requests.
    """

    if not image_url:
        return ""

    product_id = safe_filename(product_id)

    if not product_id:
        return ""

    # Reuse an already downloaded image.
    for filename in os.listdir(IMAGE_DIR):
        if filename.startswith(product_id + "."):
            return os.path.join(IMAGE_DIR, filename)

    referers = []

    if product_url:
        referers.append(product_url)

    referers.extend([
        "https://www.dtsupply.us/",
        "https://dtsupply.us/",
        "https://api.dtsupply.us/",
    ])

    referers = list(dict.fromkeys(referers))

    for referer in referers:

        for attempt in range(1, RETRIES + 1):

            try:
                image_headers = {
                    "User-Agent": HEADERS["User-Agent"],
                    "Accept": (
                        "image/avif,image/webp,image/apng,"
                        "image/svg+xml,image/*,*/*;q=0.8"
                    ),
                    "Accept-Language": "en-US,en;q=0.9",
                    "Referer": referer,
                }

                response = session.get(
                    image_url,
                    headers=image_headers,
                    timeout=TIMEOUT,
                    allow_redirects=True,
                )

                # Success
                if response.status_code == 200:
                    content_type = (
                        response.headers.get("Content-Type") or ""
                    ).lower()

                    # Never save an XML/HTML AccessDenied page.
                    if (
                        "xml" in content_type
                        or "html" in content_type
                    ):
                        break

                    if not response.content:
                        break

                    extension = get_image_extension(
                        response,
                        image_url,
                    )

                    filename = (
                        f"{product_id}{extension}"
                    )

                    filepath = os.path.join(
                        IMAGE_DIR,
                        filename,
                    )

                    with open(filepath, "wb") as image_file:
                        image_file.write(response.content)

                    time.sleep(IMAGE_DELAY)

                    return filepath

                # Retry temporary / protection responses.
                if response.status_code in (
                    403,
                    429,
                    500,
                    502,
                    503,
                    504,
                ):
                    if attempt < RETRIES:
                        time.sleep(attempt)
                        continue

                break

            except requests.RequestException as e:
                if attempt == RETRIES:
                    print(
                        f"    [image error] "
                        f"product {product_id}: {e}"
                    )
                else:
                    time.sleep(attempt)

    print(
        f"    [image failed] "
        f"product={product_id} "
        f"status={getattr(response, 'status_code', 'N/A')}"
    )

    return ""


In [5]:
def load_done_product_ids():
    """Read product IDs already saved in the CSV."""

    done = set()

    if not os.path.exists(OUTPUT_CSV):
        return done

    with open(
        OUTPUT_CSV,
        "r",
        newline="",
        encoding="utf-8",
    ) as csv_file:

        reader = csv.DictReader(csv_file)

        for row in reader:
            product_id = str(
                row.get("product_id", "")
            ).strip()

            if product_id:
                done.add(product_id)

    return done


def ensure_csv():
    """Create the CSV and header if it does not exist."""

    if not os.path.exists(OUTPUT_CSV):
        with open(
            OUTPUT_CSV,
            "w",
            newline="",
            encoding="utf-8",
        ) as csv_file:

            writer = csv.DictWriter(
                csv_file,
                fieldnames=FIELDNAMES,
            )

            writer.writeheader()


def save_product(row):
    """Append one product immediately and flush to disk."""

    ensure_csv()

    with open(
        OUTPUT_CSV,
        "a",
        newline="",
        encoding="utf-8",
    ) as csv_file:

        writer = csv.DictWriter(
            csv_file,
            fieldnames=FIELDNAMES,
            extrasaction="ignore",
        )

        writer.writerow({
            field: row.get(field, "")
            for field in FIELDNAMES
        })

        # Immediately write data to disk.
        csv_file.flush()
        os.fsync(csv_file.fileno())


print("CSV helper functions ready.")


CSV helper functions ready.


In [6]:
def scrape_all():
    """Scrape categories -> products -> images, one product at a time."""

    ensure_csv()

    done_ids = load_done_product_ids()

    print(
        f"Already saved: {len(done_ids)} products"
    )

    print("\nFetching categories...")

    categories = get_json(CATEGORIES_URL)

    if not categories:
        raise RuntimeError(
            "Could not load categories."
        )

    print(
        f"Found {len(categories)} categories."
    )

    scraped_count = len(done_ids)
    skipped_count = 0
    image_success = 0
    image_failed = 0

    # ========================================================
    # CATEGORY BY CATEGORY
    # ========================================================

    for category_number, category in enumerate(
        categories,
        start=1,
    ):

        category_id = category.get(
            "category_id"
        )

        category_name = category.get(
            "name",
            f"category-{category_id}",
        )

        print("\n" + "=" * 70)
        print(
            f"CATEGORY {category_number}/{len(categories)}"
        )
        print(f"Name: {category_name}")
        print(f"ID:   {category_id}")
        print("=" * 70)

        products = get_json(
            PRODUCTS_BY_CATEGORY_URL,
            params={"id": category_id},
        )

        if not products:
            print("No products found. Skipping category.")
            continue

        print(
            f"Products in category: {len(products)}"
        )

        # ====================================================
        # PRODUCT BY PRODUCT
        # ====================================================

        for product_number, product in enumerate(
            products,
            start=1,
        ):

            product_id = str(
                product.get("id", "")
            ).strip()

            title = product.get(
                "name",
                "",
            )

            print(
                f"\n  [{product_number}/{len(products)}] "
                f"{title}"
            )

            print(
                f"  Product ID: {product_id}"
            )

            # Resume support
            if (
                product_id
                and product_id in done_ids
            ):
                print("  -> Already saved. Skipping.")
                skipped_count += 1
                continue

            image_url = product.get(
                "imageUrl",
                "",
            ) or ""

            product_url = product.get(
                "productUrl",
                "",
            ) or ""

            # ------------------------------------------------
            # Download image FIRST
            # ------------------------------------------------

            print("  -> Downloading image...")

            image_local = download_image(
                image_url=image_url,
                product_id=product_id,
                product_url=product_url,
            )

            if image_local:
                image_success += 1
                print(
                    f"  -> Image saved: {image_local}"
                )
            elif image_url:
                image_failed += 1
                print("  -> Image could not be downloaded.")
            else:
                print("  -> No image URL.")

            # ------------------------------------------------
            # Build COMPLETE product record
            # ------------------------------------------------

            row = {
                "category": category_name,
                "category_id": category_id,
                "product_id": product_id,
                "sku": product.get("sku", ""),
                "title": title,
                "description": specs_to_text(
                    product.get("specs")
                ),
                "price": format_price(
                    product.get("price")
                ),
                "in_stock": product.get(
                    "isInStock",
                    "",
                ),
                "product_url": product_url,
                "image_url": image_url,
                "image_local": image_local,
            }

            # ------------------------------------------------
            # SAVE PRODUCT IMMEDIATELY
            # ------------------------------------------------

            save_product(row)

            if product_id:
                done_ids.add(product_id)

            scraped_count += 1

            print(
                f"  -> Product saved to CSV "
                f"({scraped_count} total)"
            )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print("\n" + "=" * 70)
    print("SCRAPING COMPLETE")
    print("=" * 70)

    print(f"Products saved:       {scraped_count}")
    print(f"Products skipped:     {skipped_count}")
    print(f"Images downloaded:    {image_success}")
    print(f"Images failed/missing:{image_failed}")
    print(
        f"CSV:                  "
        f"{os.path.abspath(OUTPUT_CSV)}"
    )
    print(
        f"Images:               "
        f"{os.path.abspath(IMAGE_DIR)}"
    )


scrape_all()


Already saved: 0 products

Fetching categories...
Found 18 categories.

CATEGORY 1/18
Name: Audio Components
ID:   11
Products in category: 168

  [1/168] 00XL284 - Lenovo 55mm Internal Speaker for ThinkCentre M710Q
  Product ID: 252
  -> Downloading image...
    [image failed] product=252 status=403
  -> Image could not be downloaded.
  -> Product saved to CSV (1 total)

  [2/168] 017PW - Dell 2-in-1 Left & Right Laptop Speaker Set for Chromebook 11
  Product ID: 154
  -> Downloading image...
    [image failed] product=154 status=403
  -> Image could not be downloaded.
  -> Product saved to CSV (2 total)

  [3/168] 029MKK - Dell Internal Speaker for OptiPlex 390 790 990 3010 7010
  Product ID: 172
  -> Downloading image...
  -> Image saved: data\images\172.webp
  -> Product saved to CSV (3 total)

  [4/168] 04N567 - Dell Desktop Speakers And Power Supply
  Product ID: 147
  -> Downloading image...
  -> Image saved: data\images\147.webp
  -> Product saved to CSV (4 total)

  [5/168] 05

PermissionError: [Errno 13] Permission denied: 'data\\dtsupply_products.csv'

In [ ]:
# ============================================================
# VERIFY RESULTS
# ============================================================

rows = []

if os.path.exists(OUTPUT_CSV):
    with open(
        OUTPUT_CSV,
        "r",
        newline="",
        encoding="utf-8",
    ) as csv_file:
        rows = list(csv.DictReader(csv_file))

downloaded = sum(
    bool(row.get("image_local"))
    for row in rows
)

missing_images = sum(
    bool(row.get("image_url"))
    and not row.get("image_local")
    for row in rows
)

print(f"Total products in CSV: {len(rows)}")
print(f"Local images:          {downloaded}")
print(f"Missing images:        {missing_images}")

print("\nFolder contents:")

for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for filename in files[:10]:
        print(f"{indent}  {filename}")

print("\nCSV columns:")
print(FIELDNAMES)
